# Offer candidates — ranked + negotiation playbook
Set `POPULATION` and run all. Produces the top-100 excel with, for every candidate:
**value** (hedonic fair-price residual), **leverage** (staleness, cuts, multi-agency, re-listing, motivated-seller wording),
**affordability** (monthly payment with your profile) and a concrete **opening / target / walk-away offer**.

Band searched: **400k–700k** (500–600k target; 400–500k flagged as renovation value-plays; hard max 620k).

In [ ]:
POPULATION = "sant_cugat"   # "sant_cugat" | "sant_quirze" | "cerdanyola"

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import analysis as an

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 90)

cfg = an.POPULATIONS[POPULATION]
data = an.load_data(POPULATION)
feats = an.build_features(data)
hed = an.fit_hedonic(feats)
feats = hed["features"]
km = an.kaplan_meier(feats)
panel = an.build_daily_panel(data["properties"], data["history"], data["events"])
market = an.compute_market_daily(panel, feats, band=an.SEARCH_BAND)
baselines = an.load_notariado()
verdict = an.market_timing_verdict(market, baselines, POPULATION)
gap = an.gap_analysis(feats, baselines, POPULATION)
profile = an.FinancialProfile()

print(f"{cfg['label']}: hedonic R²={hed['r2']:.2f}, buyer score={verdict['buyer_score']}")

## Ranked candidates (deduped: one row per physical property)

In [ ]:
ranked = an.score_candidates(feats, profile)
ranked = an.add_offer_columns(ranked, km, verdict, gap, profile)
ranked["staleness_pctile"] = ranked["days_online_effective"].map(lambda d: round(an.staleness_percentile(km, d), 2))

display_cols = ["title", "source", "price", "sqm", "rooms", "bathrooms",
                "days_online_effective", "staleness_pctile", "n_cuts", "cum_discount_pct",
                "multi_listed", "relist_count", "n_reactivations", "kw_any_motivated", "kw_renovation",
                "hedonic_fair_price", "hedonic_residual_pct",
                "value_score", "leverage_score", "payment_fit", "reno_opportunity", "final_score",
                "est_margin_pct", "offer_opening", "offer_target", "offer_walkaway", "bucket",
                "monthly_payment_needed", "affordability_verdict", "cluster_size", "n_sources", "url"]
display_cols = [c for c in display_cols if c in ranked.columns]
top100 = ranked.head(100)
display(top100[display_cols].head(25))

## Renovation value-plays (400–500k + works)

In [ ]:
reno = ranked[(ranked["reno_opportunity"] == 1)].copy()
if len(reno):
    reno["all_in_estimate"] = reno["offer_target"] + an.RENO_BUDGET
    display(reno[["title", "price", "sqm", "hedonic_residual_pct", "offer_target",
                  "all_in_estimate", "est_margin_pct", "bucket", "url"]].head(15))
    display(Markdown("`all_in_estimate` = target offer + 100k renovation. Compare against what "
                     "renovated equivalents ask (hedonic fair price of similar sqm) — if the spread "
                     "is >15% you are buying equity."))
else:
    print("No renovation candidates in the 400-500k band right now.")

## Multi-agency & motivated sellers (fastest negotiation wins)

In [ ]:
hot = ranked[(ranked["multi_listed"] == 1) | (ranked["kw_any_motivated"] == 1) | (ranked["relist_count"] >= 1)]
kw_cols = [c for c in ranked.columns if c.startswith("kw_") and c != "kw_any_motivated"]
display(hot[["title", "price", "days_online_effective", "n_cuts", "multi_listed", "relist_count"]
            + kw_cols + ["est_margin_pct", "offer_opening", "bucket", "url"]].head(20))

## Export excel

In [ ]:
out_path = f"{POPULATION}_offer_candidates_400_700k.xlsx"
export = top100[display_cols].copy()
export.to_excel(out_path, index=False)
print(f"Wrote {len(export)} candidates to {out_path}")

---
**How to use the offer columns when you visit:**
- `offer_opening` — your first bid. Never open above it; you can only move up.
- `offer_target` — where the data says the deal should land. Walking from opening→target in 2 steps max signals seriousness.
- `offer_walkaway` — hard stop (capped at 620k and at your financing reality). If they won't meet it, the next stale listing is 3 weeks away.
- `rationale` column (in the excel) lists exactly *why* the margin estimate is what it is — use those facts out loud in the negotiation ("lleva 4 meses publicado, ya bajó dos veces, está en tres agencias…").